这里计算所有hindcast的数据并且进行保存，JFM FMA MAM AMJ等等。然后不同的hindcast有不同的seasons。保存为Z3  LAT LON 平均场，数据结构是 member，season，lat，lon可以任意一个Z高度。只需要指定target_plev_hpa

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockA — Hindcast Z(lat,lon) per member seasonal means (case-wise NetCDF)
========================================================================

相比原版：
- 目标压层不再写死 Z500，而是通过 TARGET_PLEV_HPA 自由设定（例如 50hPa）
- 输出变量名与文件名自动包含目标层：Z_50hPa_season / Z50_fields_<CASE>.nc
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import xarray as xr
# ===================== 关键改造：目标层自由设定 =====================
TARGET_PLEV_HPA = 500.0                 # <- 你这次要 50 hPa（原来是 500 hPa）
PLEV_TARGET_PA = TARGET_PLEV_HPA * 100.0

# 输出命名（避免还叫 Z500）
LEVEL_TAG = f"{int(TARGET_PLEV_HPA)}" if float(TARGET_PLEV_HPA).is_integer() else f"{TARGET_PLEV_HPA}".replace(".", "p")

# ===================== 配置区 =====================
HINDCAST_BASE_DIR = Path("/home/weiji/restart_exam/hindcast_data")

# 输出目录也按目标层命名，保证后续 BlockB/plot 能找到
OUT_DIR = Path(f"/home/weiji/restart_exam/code/2060202HINDCAST/result/Z{LEVEL_TAG}_fields")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 只处理某些年份；None 表示全部年份
TARGET_YEARS: Optional[List[str]] = ["0003", "0008", "0013", "0014", "0019"]

# 变量目录/变量名
VAR_DIRNAME = "Z3"
VAR_NAME = "Z3"


VAR_OUT_NAME = f"Z_{LEVEL_TAG}hPa_season"      # e.g., Z_50hPa_season
FILE_PREFIX = f"Z{LEVEL_TAG}_fields"           # e.g., Z50_fields

# seasons (3-month windows)
SEASONS: Dict[str, Tuple[int, int, int]] = {
    "JFM": (1, 2, 3),
    "FMA": (2, 3, 4),
    "MAM": (3, 4, 5),
    "AMJ": (4, 5, 6),
}
SEASON_ORDER = ["JFM", "FMA", "MAM", "AMJ"]

# NetCDF 压缩设置
COMPRESS_LEVEL = 4
USE_FLOAT32 = True
# ==================================================


MONTH_NAME = {
    "01": "Jan", "02": "Feb", "03": "Mar", "04": "Apr",
    "05": "May", "06": "Jun", "07": "Jul", "08": "Aug",
    "09": "Sep", "10": "Oct", "11": "Nov", "12": "Dec",
}

BASE_FAMILIES: Dict[str, Dict[str, str]] = {
    "":   {"family": "small_perturbation", "short": "SP"},
    "V2": {"family": "large_perturbation", "short": "LP"},
    "V3": {"family": "q_perturbation",     "short": "QP"},
}


@dataclass
class CaseMeta:
    case_name: str
    year_str: str
    month_str: str
    suffix: str
    family: str
    family_short: str
    group_label: str


def parse_case_name(case_name: str) -> CaseMeta:
    m = re.match(r"^(?P<year>\d{4})-(?P<month>\d{2})(?P<suffix>.*)$", case_name)
    if not m:
        raise ValueError(f"Bad case name: {case_name}")

    year_str = m.group("year")
    month_str = m.group("month")
    suffix = m.group("suffix")

    month_label = MONTH_NAME.get(month_str, month_str)

    family = "unknown"
    family_short = "UNK"
    if suffix in BASE_FAMILIES:
        family = BASE_FAMILIES[suffix]["family"]
        family_short = BASE_FAMILIES[suffix]["short"]
    elif "nocouple" in suffix.lower():
        family = "initial_prescribed_o3"
        family_short = "NC"

    if family == "small_perturbation" and "nocouple" not in suffix.lower():
        group_label = f"{month_label}_couple"
    elif family == "initial_prescribed_o3":
        group_label = f"{month_label}_initial_prescribeO3"
    elif family == "large_perturbation":
        group_label = f"{month_label}_largePert"
    elif family == "q_perturbation":
        group_label = f"{month_label}_Qpert"
    else:
        group_label = case_name

    return CaseMeta(
        case_name=case_name,
        year_str=year_str,
        month_str=month_str,
        suffix=suffix,
        family=family,
        family_short=family_short,
        group_label=group_label,
    )


def list_case_dirs(base_dir: Path) -> List[Path]:
    out = []
    for p in sorted(base_dir.iterdir()):
        if p.is_dir() and re.match(r"^\d{4}-\d{2}.*$", p.name):
            out.append(p)
    return out


def list_member_files(case_dir: Path) -> List[Path]:
    vdir = case_dir / VAR_DIRNAME
    if not vdir.is_dir():
        return []
    return sorted(vdir.glob(f"*.{VAR_NAME}.nc"))


def parse_member_id_from_filename(f: Path, case_name: str, fallback_idx: int) -> int:
    parts = f.name.split(".")
    if case_name in parts:
        i = parts.index(case_name)
        if i + 1 < len(parts) and parts[i + 1].isdigit():
            try:
                return int(parts[i + 1])
            except Exception:
                pass
    return fallback_idx


def unify_vertical_dim(da: xr.DataArray) -> xr.DataArray:
    if ("plev_2" in da.dims) and ("plev" not in da.dims):
        da = da.rename({"plev_2": "plev"})
    if ("lev" in da.dims) and ("plev" not in da.dims):
        da = da.rename({"lev": "plev"})
    if ("level" in da.dims) and ("plev" not in da.dims):
        da = da.rename({"level": "plev"})
    if ("plev" in da.dims) and ("plev_2" in da.dims):
        da = da.isel(plev_2=0, drop=True)
    return da


def load_case_z3(case_dir: Path) -> Tuple[Optional[xr.DataArray], Optional[CaseMeta]]:
    case_name = case_dir.name
    meta = parse_case_name(case_name)

    files = list_member_files(case_dir)
    if not files:
        return None, None

    da_list = []
    for idx, f in enumerate(files, start=1):
        ds = xr.open_dataset(f, decode_times=True)
        if VAR_NAME not in ds.data_vars:
            ds.close()
            continue

        da = unify_vertical_dim(ds[VAR_NAME])
        ds.close()

        mid = parse_member_id_from_filename(f, case_name, fallback_idx=idx)
        da = da.expand_dims("member").assign_coords(member=[mid])
        da_list.append(da)

    if not da_list:
        return None, None

    da_all = xr.concat(da_list, dim="member", join="outer")
    da_all = da_all.sortby("member")
    return da_all, meta


def finite_time_mask(z_m: xr.DataArray) -> xr.DataArray:
    dims_to_any = [d for d in z_m.dims if d != "time"]
    return xr.apply_ufunc(np.isfinite, z_m).any(dim=dims_to_any)


def member_season_mean_field(
    z_m: xr.DataArray, months: Tuple[int, int, int]
) -> Tuple[bool, xr.DataArray, Tuple[int, int, int]]:
    valid_t = finite_time_mask(z_m)
    mon = z_m["time"].dt.month

    m1, m2, m3 = months
    c1 = int(((mon == m1) & valid_t).sum().values)
    c2 = int(((mon == m2) & valid_t).sum().values)
    c3 = int(((mon == m3) & valid_t).sum().values)

    ok = (c1 > 0) and (c2 > 0) and (c3 > 0)
    if not ok:
        nan_field = xr.full_like(z_m.isel(time=0, drop=True), np.nan)
        return False, nan_field, (c1, c2, c3)

    mask_all = ((mon == m1) | (mon == m2) | (mon == m3)) & valid_t
    mean_field = z_m.where(mask_all, drop=True).mean("time")
    return True, mean_field, (c1, c2, c3)


def describe_lengths(lengths: np.ndarray) -> str:
    if lengths.size == 0:
        return "empty"
    p25 = int(np.percentile(lengths, 25))
    p75 = int(np.percentile(lengths, 75))
    return f"min={lengths.min()} p25={p25} median={int(np.median(lengths))} p75={p75} max={lengths.max()}"


def main():
    case_dirs = list_case_dirs(HINDCAST_BASE_DIR)

    if TARGET_YEARS is not None:
        target_set = set(TARGET_YEARS)
        case_dirs = [p for p in case_dirs if p.name.split("-")[0] in target_set]

    print(f"[INFO] Found {len(case_dirs)} cases under {HINDCAST_BASE_DIR} (years={TARGET_YEARS})")
    print(f"[INFO] Output dir: {OUT_DIR}")
    print(f"[INFO] Target level: {TARGET_PLEV_HPA} hPa ({PLEV_TARGET_PA} Pa)")

    for case_dir in case_dirs:
        case_name = case_dir.name
        out_path = OUT_DIR / f"{FILE_PREFIX}_{case_name}.nc"

        if out_path.exists():
            print(f"[SKIP] Exists: {out_path}")
            continue

        da_z3, meta = load_case_z3(case_dir)
        if da_z3 is None or meta is None:
            print(f"[SKIP] {case_name}: no valid Z3 member files.")
            continue

        if "plev" not in da_z3.dims:
            print(f"[WARN] {case_name}: no 'plev' dimension found. Skip.")
            continue

        # 选目标层（nearest）
        zlev = da_z3.sel(plev=PLEV_TARGET_PA, method="nearest")  # (member,time,lat,lon)

        member_vals = zlev["member"].values
        lengths = []
        for mid in member_vals:
            z_m = zlev.sel(member=mid)
            valid_t = finite_time_mask(z_m)
            lengths.append(int(valid_t.sum().values))
        lengths_arr = np.asarray(lengths, dtype=int)

        print("\n" + "=" * 70)
        print(f"[CASE] {case_name}  group={meta.group_label}  family={meta.family_short}")
        print(f"  members={len(member_vals)}")
        print(f"  time-length distribution (n_valid_timesteps): {describe_lengths(lengths_arr)}")

        seasons = np.array(SEASON_ORDER, dtype="U3")
        nmem = len(member_vals)
        nsea = len(seasons)

        lat = zlev["lat"]
        lon = zlev["lon"] if "lon" in zlev.coords else None

        # Z_level_season: (member, season, lat, lon)
        if lon is not None:
            z_season = xr.DataArray(
                np.full((nmem, nsea, lat.size, lon.size), np.nan,
                        dtype=np.float32 if USE_FLOAT32 else np.float64),
                dims=("member", "season", "lat", "lon"),
                coords={"member": member_vals, "season": seasons, "lat": lat, "lon": lon},
                name=VAR_OUT_NAME,
            )
        else:
            z_season = xr.DataArray(
                np.full((nmem, nsea, lat.size), np.nan,
                        dtype=np.float32 if USE_FLOAT32 else np.float64),
                dims=("member", "season", "lat"),
                coords={"member": member_vals, "season": seasons, "lat": lat},
                name=VAR_OUT_NAME,
            )

        season_valid = xr.DataArray(
            np.zeros((nmem, nsea), dtype=np.int8),
            dims=("member", "season"),
            coords={"member": member_vals, "season": seasons},
            name="season_valid",
        )

        month_counts = xr.DataArray(
            np.zeros((nmem, nsea, 3), dtype=np.int16),
            dims=("member", "season", "mindex"),
            coords={"member": member_vals, "season": seasons, "mindex": [0, 1, 2]},
            name="month_counts",
        )

        season_months = xr.DataArray(
            np.array([SEASONS[s] for s in SEASON_ORDER], dtype=np.int8),
            dims=("season", "mindex"),
            coords={"season": seasons, "mindex": [0, 1, 2]},
            name="season_months",
        )

        valid_members_count = {s: 0 for s in SEASON_ORDER}

        for im, mid in enumerate(member_vals):
            z_m = zlev.sel(member=mid)  # (time,lat,lon)
            for isea, s in enumerate(SEASON_ORDER):
                months = SEASONS[s]
                ok, mean_field, (c1, c2, c3) = member_season_mean_field(z_m, months)

                month_counts[im, isea, 0] = c1
                month_counts[im, isea, 1] = c2
                month_counts[im, isea, 2] = c3

                if ok:
                    season_valid[im, isea] = 1
                    valid_members_count[s] += 1
                    if lon is not None:
                        z_season[im, isea, :, :] = mean_field.astype(z_season.dtype)
                    else:
                        z_season[im, isea, :] = mean_field.astype(z_season.dtype)

        for s in SEASON_ORDER:
            print(f"  season {s}: valid_members={valid_members_count[s]}/{nmem}")

        ds_out = xr.Dataset(
            data_vars={
                VAR_OUT_NAME: z_season,
                "season_valid": season_valid,
                "month_counts": month_counts,
                "season_months": season_months,
            },
            attrs={
                "case_name": meta.case_name,
                "year_str": meta.year_str,
                "month_str": meta.month_str,
                "suffix": meta.suffix,
                "family": meta.family,
                "family_short": meta.family_short,
                "group_label": meta.group_label,
                "plev_target_pa": float(PLEV_TARGET_PA),
                "plev_target_hpa": float(TARGET_PLEV_HPA),
                "season_rule": (
                    "A member-season is computed only if each of the three calendar months "
                    "has >=1 valid timestep (not all-NaN over lat/lon). Otherwise NaN."
                ),
                "note": (
                    f"Per-case output. Contains per-member seasonal mean Z at {TARGET_PLEV_HPA} hPa "
                    "for JFM/FMA/MAM/AMJ. Missing seasons for a member are NaN with season_valid=0."
                ),
            },
        )

        # 单位继承
        if "units" in zlev.attrs:
            ds_out[VAR_OUT_NAME].attrs["units"] = zlev.attrs["units"]

        encoding = {
            VAR_OUT_NAME: {"zlib": True, "complevel": COMPRESS_LEVEL},
            "season_valid": {"zlib": True, "complevel": COMPRESS_LEVEL},
            "month_counts": {"zlib": True, "complevel": COMPRESS_LEVEL},
            "season_months": {"zlib": True, "complevel": COMPRESS_LEVEL},
        }

        ds_out.to_netcdf(out_path, encoding=encoding)
        print(f"[SAVE] -> {out_path}")

    print("\n[INFO] All done.")


if __name__ == "__main__":
    main()


这里处理气候态BW2000CN和longrun的数据

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, glob
import numpy as np
import xarray as xr

# ============================================================
#  Build reference Z(level) seasonal means:
#   (A) Climatology from B2000WCN-withchem extracted hybrid Z3 yearly files
#       -> monthly mean -> stdplev -> 12-month climatology -> seasonal means
#   (B) Longrun BWCN.002 h3 history files
#       -> monthly mean -> stdplev -> seasonal means per-year
#
#  Seasons: JFM, FMA, MAM, AMJ
#  Target level: configurable by TARGET_PLEV_HPA (e.g., 50 hPa)
# ============================================================

# ===================== target level =========================
TARGET_PLEV_HPA = 500.0
PLEV_TARGET_PA  = TARGET_PLEV_HPA * 100.0

LEVEL_TAG = f"{int(TARGET_PLEV_HPA)}" if float(TARGET_PLEV_HPA).is_integer() else f"{TARGET_PLEV_HPA}".replace(".", "p")
VAR_TAG   = f"Z{LEVEL_TAG}"              # e.g., Z50

# ======================== paths =============================
OUT_DIR = f"/home/weiji/restart_exam/code/2060202HINDCAST/result/{VAR_TAG}_fields"
os.makedirs(OUT_DIR, exist_ok=True)

# (A) climatology source (B2000WCN)
B2000_Z3_HYB_DIR = "/home/weiji/restart_exam/longrun_B2000WCN_withchem_data/Z3"
OUT_CLIM_SEASON  = os.path.join(OUT_DIR, f"{VAR_TAG}_climatology_season_12mon.nc")

# (B) longrun source (BWCN)
BWCN_H3_DIR      = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"
OUT_LONGRUN_SEAS = os.path.join(OUT_DIR, f"{VAR_TAG}_longrun_season_yearly.nc")

KEEP_MODEL_YEARS = [3, 8, 13, 14, 19]  # or None

# ===================== standard plev =========================
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

# ===================== seasons ===============================
SEASONS = {
    "JFM": (1, 2, 3),
    "FMA": (2, 3, 4),
    "MAM": (3, 4, 5),
    "AMJ": (4, 5, 6),
}
SEASON_ORDER = ["JFM", "FMA", "MAM", "AMJ"]

# ===================== IO strategy ===========================
ENGINE   = "netcdf4"
PARALLEL = False
CHUNKS   = {"time": 1}

COMPRESS_LEVEL = 4
USE_FLOAT32 = True

# =============================================================
def safe_open_mfdataset(files):
    """Default: by_coords (OK for BWCN)."""
    return xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=PARALLEL,
        use_cftime=True,
        chunks=CHUNKS,
        engine=ENGINE,
    )

def safe_open_concat_time(files):
    """
    For B2000WCN parallel segments where 'time' can overlap.
    Key: DO NOT align by 'time' coords. Just concat in file order.
    """
    dsets = []
    for f in files:
        ds = xr.open_dataset(f, engine=ENGINE, chunks=CHUNKS, use_cftime=True)
        dsets.append(ds)

    # concat directly on time; do NOT merge/align by coords
    ds_all = xr.concat(
        dsets,
        dim="time",
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )

    # best-effort close original handles (concat keeps refs)
    for ds in dsets:
        try:
            ds.close()
        except Exception:
            pass

    return ds_all

def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]  # (lev)
    hybm = ds["hybm"]  # (lev)
    P0   = ds["P0"]    # scalar Pa
    PS   = ds["PS"]    # (time, lat, lon)
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def interp_profile_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt: np.ndarray) -> xr.DataArray:
    p_tgt = np.asarray(p_tgt, float)
    nplev = int(p_tgt.size)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full((nplev,), np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)  # ascending pressure
        p_use = p_use[idx]
        v_use = v_use[idx]
        return np.interp(np.log(p_tgt), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={"plev": nplev},
    )
    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out

def monthly_mean_then_interp_to_stdplev(ds: xr.Dataset, varname="Z3") -> xr.DataArray:
    need = [varname, "PS", "hyam", "hybm", "P0"]
    missing = [v for v in need if v not in ds.variables]
    if missing:
        raise RuntimeError(f"Dataset missing vars: {missing}")

    ds_m = ds[need].resample(time="MS").mean()
    v_m = ds_m[varname]               # (time, lev, lat, lon)
    p_m = compute_pressure_mid(ds_m)  # (time, lev, lat, lon)

    v_std = interp_profile_logp(v_m, p_m, PLEV_STD_PA)
    v_std = v_std.transpose("time", "plev", "lat", "lon")

    v_std.name = f"{varname}_stdplev"
    v_std.attrs.update(
        units=v_m.attrs.get("units", ""),
        long_name=f"{varname}: monthly mean then log(p) interpolated to standard pressure levels",
    )
    return v_std

def seasonal_mean_from_monthly_field(v_monthly: xr.DataArray, months: tuple) -> xr.DataArray:
    m1, m2, m3 = months
    mon = v_monthly["time"].dt.month
    yr  = v_monthly["time"].dt.year

    out_list = []
    years = np.unique(yr.values)

    for y in years:
        sel = v_monthly.where(yr == y, drop=True)
        sel_mon = sel["time"].dt.month
        c1 = int((sel_mon == m1).sum().values)
        c2 = int((sel_mon == m2).sum().values)
        c3 = int((sel_mon == m3).sum().values)

        if (c1 > 0) and (c2 > 0) and (c3 > 0):
            msel = sel.where((sel_mon == m1) | (sel_mon == m2) | (sel_mon == m3), drop=True)
            mean_y = msel.mean("time")
        else:
            mean_y = xr.full_like(sel.isel(time=0, drop=True), np.nan)

        mean_y = mean_y.expand_dims("year").assign_coords(year=[int(y)])
        out_list.append(mean_y)

    out = xr.concat(out_list, dim="year")
    return out

def write_dataset(ds: xr.Dataset, out_path: str, compress_level=4):
    enc = {}
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating):
            enc[v] = {"zlib": True, "complevel": int(compress_level),
                      "dtype": "float32" if USE_FLOAT32 else str(ds[v].dtype),
                      "_FillValue": np.float32(np.nan)}
        else:
            enc[v] = {"zlib": True, "complevel": int(compress_level)}
    for c in ds.coords:
        if c in enc:
            continue
        if ds[c].dtype.kind in ["f"]:
            enc[c] = {"dtype": "float64"}
        elif ds[c].dtype.kind in ["i", "u"]:
            enc[c] = {"dtype": "int32"}
    ds.to_netcdf(out_path, encoding=enc)

# =========================================================
# (A) Climatology: 12-month climatology -> seasonal means
# =========================================================
print("[A] Building climatology seasonal means from:", B2000_Z3_HYB_DIR)

b2000_files = sorted(glob.glob(os.path.join(B2000_Z3_HYB_DIR, "*.Z3.hybrid.nc")))
if len(b2000_files) == 0:
    raise RuntimeError(f"No B2000 hybrid Z3 files found: {B2000_Z3_HYB_DIR}")

# ✅ changed: for B2000WCN, concat by time (avoid overlap alignment)
ds_b2000 = safe_open_concat_time(b2000_files)

# optional diagnostic: check time uniqueness
try:
    t = ds_b2000["time"].to_index()
    if not t.is_unique:
        print(f"[WARN] B2000WCN time is not unique after concat (this is expected for parallel segments). "
              f"Proceeding because climatology uses month-of-year only.")
except Exception:
    pass

z3_b2000_monthly_std = monthly_mean_then_interp_to_stdplev(ds_b2000, varname="Z3")  # (time,plev,lat,lon)
try:
    ds_b2000.close()
except Exception:
    pass

zlev_b2000_monthly = z3_b2000_monthly_std.sel(plev=PLEV_TARGET_PA, method="nearest")  # (time,lat,lon)

# 12-month climatology (month_of_year=1..12)
zlev_clim12 = zlev_b2000_monthly.groupby("time.month").mean("time").rename({"month": "month_of_year"})
zlev_clim12["month_of_year"].attrs["long_name"] = "Month of year (1-12)"

# seasonal mean using climatology months
season_labels = np.array(SEASON_ORDER, dtype="U3")
clim_season = []
for s in SEASON_ORDER:
    m1, m2, m3 = SEASONS[s]
    mean_s = zlev_clim12.sel(month_of_year=[m1, m2, m3]).mean("month_of_year")
    mean_s = mean_s.expand_dims("season").assign_coords(season=[s])
    clim_season.append(mean_s)

Zlev_clim_season = xr.concat(clim_season, dim="season")  # (season,lat,lon)

clim_varname = f"{VAR_TAG}_clim_season"                  # e.g., Z50_clim_season

ds_clim_out = xr.Dataset(
    data_vars={clim_varname: Zlev_clim_season.astype(np.float32 if USE_FLOAT32 else np.float64)},
    coords={"season": season_labels, "lat": Zlev_clim_season["lat"], "lon": Zlev_clim_season["lon"]},
    attrs={
        "source": "B2000WCN_withchem longrun extracted hybrid Z3",
        "processing": (
            f"monthly mean -> stdplev log(p) interp -> Z{TARGET_PLEV_HPA:g}hPa "
            "-> 12-month climatology -> 3-month seasonal means"
        ),
        "seasons": "JFM,FMA,MAM,AMJ",
        "plev_target_pa": float(PLEV_TARGET_PA),
        "plev_target_hpa": float(TARGET_PLEV_HPA),
        "note": "B2000WCN merged by direct time-concat to avoid by_coords alignment on overlapping time.",
    },
)

if "units" in z3_b2000_monthly_std.attrs:
    ds_clim_out[clim_varname].attrs["units"] = z3_b2000_monthly_std.attrs["units"]

write_dataset(ds_clim_out, OUT_CLIM_SEASON, compress_level=COMPRESS_LEVEL)
print("[SAVE] climatology seasons ->", OUT_CLIM_SEASON)

# =========================================================
# (B) Longrun: yearly seasonal means from BWCN h3
# =========================================================
print("\n[B] Building longrun seasonal means from:", BWCN_H3_DIR)

bwcn_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
if len(bwcn_files) == 0:
    raise RuntimeError(f"No BWCN h3 files found: {BWCN_H3_DIR}")

# keep BWCN as by_coords (you said BWCN has no issue)
ds_bwcn = safe_open_mfdataset(bwcn_files)
z3_bwcn_monthly_std = monthly_mean_then_interp_to_stdplev(ds_bwcn, varname="Z3")  # (time,plev,lat,lon)
ds_bwcn.close()

zlev_bwcn_monthly = z3_bwcn_monthly_std.sel(plev=PLEV_TARGET_PA, method="nearest")  # (time,lat,lon)

# optionally filter years
if KEEP_MODEL_YEARS is not None:
    years_all = zlev_bwcn_monthly["time"].dt.year
    zlev_bwcn_monthly = zlev_bwcn_monthly.where(years_all.isin(KEEP_MODEL_YEARS), drop=True)

# compute yearly seasonal means for each season
season_fields = []
for s in SEASON_ORDER:
    seas_y = seasonal_mean_from_monthly_field(zlev_bwcn_monthly, SEASONS[s])  # (year,lat,lon)
    seas_y = seas_y.expand_dims("season").assign_coords(season=[s])
    season_fields.append(seas_y)

Zlev_longrun_season = xr.concat(season_fields, dim="season")  # (season,year,lat,lon)
Zlev_longrun_season = Zlev_longrun_season.transpose("year", "season", "lat", "lon")

longrun_varname = f"{VAR_TAG}_longrun_season"                 # e.g., Z50_longrun_season

ds_longrun_out = xr.Dataset(
    data_vars={longrun_varname: Zlev_longrun_season.astype(np.float32 if USE_FLOAT32 else np.float64)},
    coords={
        "year": Zlev_longrun_season["year"].astype(np.int32),
        "season": season_labels,
        "lat": Zlev_longrun_season["lat"],
        "lon": Zlev_longrun_season["lon"],
    },
    attrs={
        "source": "BWCN.e122.f19_g16.002 atm/hist cam.h3",
        "processing": (
            f"monthly mean -> stdplev log(p) interp -> Z{TARGET_PLEV_HPA:g}hPa "
            "-> yearly 3-month seasonal means with completeness check"
        ),
        "seasons": "JFM,FMA,MAM,AMJ",
        "plev_target_pa": float(PLEV_TARGET_PA),
        "plev_target_hpa": float(TARGET_PLEV_HPA),
        "year_filter": str(KEEP_MODEL_YEARS),
        "season_rule": "For a given (year, season), must contain all 3 calendar months; otherwise NaN.",
    },
)

if "units" in z3_bwcn_monthly_std.attrs:
    ds_longrun_out[longrun_varname].attrs["units"] = z3_bwcn_monthly_std.attrs["units"]

write_dataset(ds_longrun_out, OUT_LONGRUN_SEAS, compress_level=COMPRESS_LEVEL)
print("[SAVE] longrun yearly seasons ->", OUT_LONGRUN_SEAS)

print("\n[INFO] Done.")


根据气候态和hindcast/longrun数据计算anomaly和对应的ACC。

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import re
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd

# ============================================================
#  BlockB — Compute anomalies + ACC (hindcast vs longrun ref)
#
#  Target level is configurable by TARGET_PLEV_HPA (e.g., 50 hPa)
#
#  Inputs:
#    - Hindcast per-case:
#        Z{L}_fields_<CASE>.nc
#        var: Z_{L}hPa_season(member,season,lat,lon), season_valid(member,season)
#    - Climatology:
#        Z{L}_climatology_season_12mon.nc
#        var: Z{L}_clim_season(season,lat,lon)
#    - Longrun:
#        Z{L}_longrun_season_yearly.nc
#        var: Z{L}_longrun_season(year,season,lat,lon)
#
#  Outputs:
#    - Hindcast anomalies:
#        Z{L}_anom_fields_<CASE>.nc
#    - Longrun anomalies:
#        Z{L}_longrun_anom_yearly.nc
#    - ACC per case:
#        ACC_<CASE>.nc  + ACC_summary.csv
#
#  ACC: cos(lat)-weighted pattern correlation over lat/lon.
# ============================================================

# ===================== target level =====================
TARGET_PLEV_HPA = 500.0                    # <- 你这次设 50hPa
PLEV_TARGET_PA  = TARGET_PLEV_HPA * 100.0

LEVEL_TAG = f"{int(TARGET_PLEV_HPA)}" if float(TARGET_PLEV_HPA).is_integer() else f"{TARGET_PLEV_HPA}".replace(".", "p")
VAR_TAG   = f"Z{LEVEL_TAG}"               # e.g., Z50

# variable names produced by previous blocks
HIND_VAR_NAME = f"Z_{LEVEL_TAG}hPa_season"     # e.g., Z_50hPa_season   (BlockA)
CLIM_VAR_NAME = f"{VAR_TAG}_clim_season"       # e.g., Z50_clim_season  (ref A)
LR_VAR_NAME   = f"{VAR_TAG}_longrun_season"    # e.g., Z50_longrun_season (ref B)

# output variable names
HIND_ANOM_NAME = f"{VAR_TAG}_hindcast_anom"    # e.g., Z50_hindcast_anom
LR_ANOM_NAME   = f"{VAR_TAG}_longrun_anom"     # e.g., Z50_longrun_anom

# ===================== paths =====================
BASE_DIR = Path(f"/home/weiji/restart_exam/code/2060202HINDCAST/result/{VAR_TAG}_fields")

HIND_DIR     = BASE_DIR
CLIM_FILE    = BASE_DIR / f"{VAR_TAG}_climatology_season_12mon.nc"
LONGRUN_FILE = BASE_DIR / f"{VAR_TAG}_longrun_season_yearly.nc"

OUT_ANOM_HIND_DIR = BASE_DIR / f"{VAR_TAG}_anom_fields"
OUT_ANOM_REF_DIR  = BASE_DIR / f"{VAR_TAG}_anom_ref"
OUT_ACC_DIR       = BASE_DIR / "ACC"

OUT_ANOM_HIND_DIR.mkdir(parents=True, exist_ok=True)
OUT_ANOM_REF_DIR.mkdir(parents=True, exist_ok=True)
OUT_ACC_DIR.mkdir(parents=True, exist_ok=True)

OUT_REF_ANOM_FILE = OUT_ANOM_REF_DIR / f"{VAR_TAG}_longrun_anom_yearly.nc"
OUT_ACC_CSV       = OUT_ACC_DIR / "ACC_summary.csv"

SEASON_ORDER = ["JFM", "FMA", "MAM", "AMJ"]

TARGET_YEARS = {"0003", "0008", "0013", "0014", "0019"}  # or None
# TARGET_YEARS = None

COMPRESS_LEVEL = 4
USE_FLOAT32 = True

# ===================== helpers =====================
def _to_float32_if(x):
    return x.astype(np.float32) if USE_FLOAT32 else x

def parse_year_from_case(case_name: str) -> int:
    m = re.match(r"^(?P<year>\d{4})-", case_name)
    if not m:
        raise ValueError(f"Cannot parse year from case_name={case_name}")
    return int(m.group("year"))

def coslat_weights(lat: xr.DataArray) -> xr.DataArray:
    w = np.cos(np.deg2rad(lat))
    w = xr.DataArray(np.maximum(w, 0.0), coords={"lat": lat}, dims=("lat",))
    return w

def weighted_corr2d(a2d: np.ndarray, b2d: np.ndarray, w_lat: np.ndarray) -> float:
    if a2d.ndim != 2 or b2d.ndim != 2:
        raise ValueError("a2d/b2d must be 2D (lat,lon).")
    lat_n, lon_n = a2d.shape
    if b2d.shape != (lat_n, lon_n):
        raise ValueError("a2d and b2d must have same shape.")

    w2d = np.repeat(w_lat.reshape(lat_n, 1), lon_n, axis=1)
    m = np.isfinite(a2d) & np.isfinite(b2d) & np.isfinite(w2d) & (w2d > 0)
    if m.sum() < 10:
        return np.nan

    aa = a2d[m]
    bb = b2d[m]
    ww = w2d[m]

    wsum = ww.sum()
    if wsum <= 0:
        return np.nan

    am = (ww * aa).sum() / wsum
    bm = (ww * bb).sum() / wsum

    a0 = aa - am
    b0 = bb - bm

    cov = (ww * a0 * b0).sum()
    va  = (ww * a0 * a0).sum()
    vb  = (ww * b0 * b0).sum()
    if va <= 0 or vb <= 0:
        return np.nan

    return float(cov / np.sqrt(va * vb))

def list_hindcast_case_files(hind_dir: Path):
    files = sorted(hind_dir.glob(f"{VAR_TAG}_fields_*.nc"))
    out = []
    for f in files:
        case = f.name.replace(f"{VAR_TAG}_fields_", "").replace(".nc", "")
        if TARGET_YEARS is not None:
            y4 = case.split("-")[0]
            if y4 not in TARGET_YEARS:
                continue
        out.append((case, f))
    return out

def write_netcdf(ds: xr.Dataset, out_path: Path):
    enc = {}
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating):
            enc[v] = {
                "zlib": True, "complevel": int(COMPRESS_LEVEL),
                "dtype": "float32" if USE_FLOAT32 else str(ds[v].dtype),
                "_FillValue": np.float32(np.nan),
            }
        else:
            enc[v] = {"zlib": True, "complevel": int(COMPRESS_LEVEL)}
    ds.to_netcdf(out_path, encoding=enc)

# ===================== main =====================
def main():
    print(f"[INFO] Target level: {TARGET_PLEV_HPA} hPa ({PLEV_TARGET_PA} Pa)")
    print(f"[INFO] BASE_DIR: {BASE_DIR}")
    print(f"[INFO] Hind var: {HIND_VAR_NAME} | Clim var: {CLIM_VAR_NAME} | Longrun var: {LR_VAR_NAME}")

    # ---- load climatology ----
    if not CLIM_FILE.exists():
        raise FileNotFoundError(f"Missing climatology file: {CLIM_FILE}")
    ds_clim = xr.open_dataset(CLIM_FILE)
    if CLIM_VAR_NAME not in ds_clim:
        raise KeyError(f"Climatology file must contain variable '{CLIM_VAR_NAME}'")
    clim = ds_clim[CLIM_VAR_NAME].sel(season=SEASON_ORDER)  # (season,lat,lon)
    lat = clim["lat"]
    lon = clim["lon"]
    w_lat = coslat_weights(lat).values
    ds_clim.close()

    # ---- load longrun ----
    if not LONGRUN_FILE.exists():
        raise FileNotFoundError(f"Missing longrun file: {LONGRUN_FILE}")
    ds_lr = xr.open_dataset(LONGRUN_FILE)
    if LR_VAR_NAME not in ds_lr:
        raise KeyError(f"Longrun file must contain variable '{LR_VAR_NAME}'")
    lr = ds_lr[LR_VAR_NAME].sel(season=SEASON_ORDER)  # (year,season,lat,lon)

    # ---- compute longrun anomalies once and save ----
    lr_anom = lr - clim
    lr_anom.name = LR_ANOM_NAME

    ds_lr_anom = xr.Dataset(
        data_vars={LR_ANOM_NAME: _to_float32_if(lr_anom)},
        coords={
            "year": lr_anom["year"].astype(np.int32),
            "season": lr_anom["season"].astype("U3"),
            "lat": lr_anom["lat"],
            "lon": lr_anom["lon"],
        },
        attrs={
            "note": f"Longrun (BWCN) seasonal anomalies at {TARGET_PLEV_HPA:g} hPa relative to B2000 climatology.",
            "climatology_file": str(CLIM_FILE),
            "longrun_file": str(LONGRUN_FILE),
            "seasons": ",".join(SEASON_ORDER),
            "plev_target_pa": float(PLEV_TARGET_PA),
            "plev_target_hpa": float(TARGET_PLEV_HPA),
        },
    )

    if OUT_REF_ANOM_FILE.exists():
        print(f"[SKIP] Ref anomaly exists: {OUT_REF_ANOM_FILE}")
    else:
        write_netcdf(ds_lr_anom, OUT_REF_ANOM_FILE)
        print(f"[SAVE] Ref anomaly -> {OUT_REF_ANOM_FILE}")

    lr_anom_inmem = lr_anom

    # ---- process each hindcast case ----
    case_files = list_hindcast_case_files(HIND_DIR)
    print(f"[INFO] Found {len(case_files)} hindcast case files in {HIND_DIR}")

    summary_rows = []

    for case_name, fcase in case_files:
        print("\n" + "=" * 80)
        print(f"[CASE] {case_name}")
        ds_h = xr.open_dataset(fcase)

        if HIND_VAR_NAME not in ds_h:
            print(f"[SKIP] {fcase} has no {HIND_VAR_NAME}")
            ds_h.close()
            continue

        z = ds_h[HIND_VAR_NAME].sel(season=SEASON_ORDER)  # (member,season,lat,lon)

        if ("lat" not in z.coords) or ("lon" not in z.coords):
            print("[SKIP] Missing lat/lon coords")
            ds_h.close()
            continue

        # align to climatology grid if needed
        z = z.reindex(lat=lat, lon=lon)

        # season_valid optional
        if "season_valid" in ds_h:
            sv = ds_h["season_valid"].sel(season=SEASON_ORDER)  # (member,season)
        else:
            sv = xr.DataArray(
                np.ones((z.sizes["member"], z.sizes["season"]), dtype=np.int8),
                dims=("member", "season"),
                coords={"member": z["member"], "season": z["season"]},
            )

        # ---- hindcast anomaly ----
        hind_anom = z - clim
        hind_anom.name = HIND_ANOM_NAME

        out_anom_case = OUT_ANOM_HIND_DIR / f"{VAR_TAG}_anom_fields_{case_name}.nc"
        if out_anom_case.exists():
            print(f"[SKIP] Hind anomaly exists: {out_anom_case}")
        else:
            ds_h_anom = xr.Dataset(
                data_vars={
                    HIND_ANOM_NAME: _to_float32_if(hind_anom),
                    "season_valid": sv.astype(np.int8),
                },
                coords={
                    "member": hind_anom["member"],
                    "season": hind_anom["season"].astype("U3"),
                    "lat": hind_anom["lat"],
                    "lon": hind_anom["lon"],
                },
                attrs={
                    "case_name": case_name,
                    "source_hindcast_file": str(fcase),
                    "climatology_file": str(CLIM_FILE),
                    "plev_target_pa": float(ds_h.attrs.get("plev_target_pa", PLEV_TARGET_PA)),
                    "plev_target_hpa": float(ds_h.attrs.get("plev_target_hpa", TARGET_PLEV_HPA)),
                    "note": f"Hindcast seasonal anomalies at {TARGET_PLEV_HPA:g} hPa relative to B2000 climatology.",
                },
            )
            write_netcdf(ds_h_anom, out_anom_case)
            print(f"[SAVE] Hind anomaly -> {out_anom_case}")

        # ---- ACC vs reference (year-aligned) ----
        y_int = parse_year_from_case(case_name)

        if "year" not in lr_anom_inmem.dims:
            print("[SKIP] longrun anomaly missing year dimension")
            ds_h.close()
            continue

        if y_int not in lr_anom_inmem["year"].values:
            print(f"[WARN] reference longrun does not contain year={y_int}. ACC will be NaN for this case.")
            ref_anom_y = None
        else:
            ref_anom_y = lr_anom_inmem.sel(year=y_int)  # (season,lat,lon)

        members = hind_anom["member"].values
        seasons = hind_anom["season"].values

        acc_member = np.full((members.size, seasons.size), np.nan, dtype=np.float32 if USE_FLOAT32 else np.float64)
        acc_ens    = np.full((seasons.size,), np.nan, dtype=np.float32 if USE_FLOAT32 else np.float64)

        for isea, s in enumerate(seasons):
            if ref_anom_y is None:
                continue

            ref2d = ref_anom_y.sel(season=s).values
            if not np.isfinite(ref2d).any():
                continue

            valid_mask = sv.sel(season=s).values.astype(bool)
            for im, mid in enumerate(members):
                if not valid_mask[im]:
                    continue
                a2d = hind_anom.sel(member=mid, season=s).values
                acc_member[im, isea] = weighted_corr2d(a2d, ref2d, w_lat)

            if valid_mask.any():
                ens2d = hind_anom.sel(season=s).where(sv.sel(season=s) == 1).mean("member").values
                acc_ens[isea] = weighted_corr2d(ens2d, ref2d, w_lat)

        out_acc_case = OUT_ACC_DIR / f"ACC_{case_name}.nc"
        ds_acc = xr.Dataset(
            data_vars={
                "ACC_member": (("member", "season"), acc_member),
                "ACC_ensmean": (("season",), acc_ens),
            },
            coords={
                "member": members,
                "season": seasons.astype("U3"),
            },
            attrs={
                "case_name": case_name,
                "hindcast_year_int": int(y_int),
                "reference_year_int": int(y_int),
                "acc_definition": "Area-weighted pattern correlation of anomalies over lat/lon using cos(lat) weights.",
                "note": f"ACC computed at {TARGET_PLEV_HPA:g} hPa between hindcast anomaly and longrun anomaly for matched model-year.",
                "hindcast_file": str(fcase),
                "climatology_file": str(CLIM_FILE),
                "reference_anom_file": str(OUT_REF_ANOM_FILE),
                "plev_target_pa": float(PLEV_TARGET_PA),
                "plev_target_hpa": float(TARGET_PLEV_HPA),
            },
        )
        write_netcdf(ds_acc, out_acc_case)
        print(f"[SAVE] ACC -> {out_acc_case}")

        for isea, s in enumerate(seasons):
            summary_rows.append({
                "case": case_name,
                "year": y_int,
                "season": str(s),
                "ACC_ensmean": float(acc_ens[isea]) if np.isfinite(acc_ens[isea]) else np.nan,
                "n_valid_members": int(sv.sel(season=s).sum().values),
                "group_label": ds_h.attrs.get("group_label", ""),
                "family_short": ds_h.attrs.get("family_short", ""),
                "suffix": ds_h.attrs.get("suffix", ""),
                "target_hpa": float(TARGET_PLEV_HPA),
            })

        ds_h.close()

    # ---- write summary CSV ----
    if summary_rows:
        df = pd.DataFrame(summary_rows)
        df = df.sort_values(["year", "case", "season"])
        df.to_csv(OUT_ACC_CSV, index=False)
        print("\n" + "=" * 80)
        print(f"[SAVE] ACC summary CSV -> {OUT_ACC_CSV}")
        print(df.head(10).to_string(index=False))
    else:
        print("[WARN] No cases processed; summary is empty.")

    ds_lr.close()
    print("\n[INFO] Done.")


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations

import re
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath


# ===================== TARGET LEVEL =====================
TARGET_PLEV_HPA = 500.0
PLEV_TARGET_PA  = TARGET_PLEV_HPA * 100.0

LEVEL_TAG = f"{int(TARGET_PLEV_HPA)}" if float(TARGET_PLEV_HPA).is_integer() else f"{TARGET_PLEV_HPA}".replace(".", "p")
VAR_TAG   = f"Z{LEVEL_TAG}"               # e.g., Z50

HIND_ANOM_VAR = f"{VAR_TAG}_hindcast_anom"
REF_ANOM_VAR  = f"{VAR_TAG}_longrun_anom"

# ===================== PATHS =====================
BASE_DIR = Path(f"/home/weiji/restart_exam/code/2060202HINDCAST/result/{VAR_TAG}_fields")

HIND_ANOM_DIR = BASE_DIR / f"{VAR_TAG}_anom_fields"
REF_ANOM_FILE = BASE_DIR / f"{VAR_TAG}_anom_ref" / f"{VAR_TAG}_longrun_anom_yearly.nc"
ACC_CSV       = BASE_DIR / "ACC" / "ACC_summary.csv"

HIND_FIELDS_DIR = BASE_DIR  # has Z{L}_fields_<CASE>.nc for attrs

OUT_FIG_DIR = BASE_DIR / f"fig_lee_style_reversexy_{VAR_TAG}"
OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

# ===================== CONFIG =====================
TARGET_YEARS = ["0003", "0008", "0013", "0014", "0019"]
SEASONS      = ["JFM", "FMA", "MAM", "AMJ"]

MIN_EXPS_PER_FIG = 2
MAX_MAPS_PER_ROW = 3

MAP_PROJ = ccrs.NorthPolarStereo(central_longitude=0)
DATA_CRS = ccrs.PlateCarree()
POLAR_EXTENT = [0, 360, 20, 90]

CMAP = "RdBu_r"
COASTLINE_LW = 0.5
BORDER_LW = 0.3

FIGSIZE = (12.5, 8.5)
FS_TITLE = 10
FS_LABEL = 9
FS_TICK  = 8

VLIM_Q = 0.98

# ---- BAR spacing controls (keep bar thickness + fonts unchanged) ----
BAR_HEIGHT = 0.60

# ✅ 拉开 y 方向间距（减少重叠）
BAR_Y_SPACING = 1.60     # 原 1.25 -> 1.60（更松）

# ✅ 给 bar panel 更多高度（更“高”）
BAR_ROW_BASE = 0.75      # 原 0.55 -> 0.75
BAR_ROW_PER_BAR = 0.060  # 原 0.035 -> 0.060


# =================================================
def normalize_season(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().upper()
    if s == "APJ":
        return "AMJ"
    return s


def safe_sel_season(da: xr.DataArray, season: str) -> xr.DataArray:
    season = normalize_season(season)
    vals = [normalize_season(x) for x in da["season"].values.tolist()]
    idx = np.where(np.array(vals) == season)[0]
    if idx.size == 0:
        raise KeyError(f"Season {season} not found. Available={vals}")
    return da.isel(season=int(idx[0]))


# ===================== NEW: label & family rules =====================
def _lower(x: str) -> str:
    return str(x).lower()

def infer_family_short_from_case(case: str) -> str:
    """
    你要求的硬规则：
    - 只要识别到含有 V2/v2 -> LP
    - 只要识别到含有 V3/v3 -> QP
    - 否则 -> SP（默认小扰动）
    """
    c = _lower(case)
    if "v2" in c:
        return "LP"
    if "v3" in c:
        return "QP"
    return "SP"

def infer_ozone_mode_from_case(case: str) -> str:
    """
    - nocouple / nocoupl -> prescribed o3
    - 其它全部 -> interactive o3
    """
    c = _lower(case)
    if ("nocouple" in c) or ("nocoupl" in c):
        return "prescribed o3"
    return "interactive o3"

def pretty_exp_name(raw: str) -> str:
    """
    把名称里的 couple/nocouple 统一换成你要的文字。
    """
    s = str(raw)

    # 先处理 nocouple（包括 NOCOUPL 这种截断）
    s = re.sub(r"(?i)nocoupl[e]?", "prescribed o3", s)

    # 再处理 couple（避免把 nocouple 里的 couple 误替换）
    # 上面已经把 nocouple 换掉了，所以这里直接替换剩余 couple
    s = re.sub(r"(?i)couple", "interactive o3", s)

    # 统一多余连接符
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def list_cases_for_year(year_str: str) -> list[str]:
    patt = f"{VAR_TAG}_anom_fields_{year_str}-*.nc"
    out = []
    for f in sorted(HIND_ANOM_DIR.glob(patt)):
        out.append(f.name.replace(f"{VAR_TAG}_anom_fields_", "").replace(".nc", ""))
    return out


def load_acc_table() -> pd.DataFrame:
    if not ACC_CSV.exists():
        raise FileNotFoundError(f"Missing ACC summary CSV: {ACC_CSV}")
    df = pd.read_csv(ACC_CSV)
    df["season"] = df["season"].astype(str).str.upper().map(normalize_season)
    df["case"] = df["case"].astype(str)
    df["year_str"] = df["year"].apply(lambda x: f"{int(x):04d}")
    return df


def load_case_meta(case: str) -> dict:
    """
    仍然读取 group_label 等 attrs，但 family_short 我们不再依赖它，
    因为你要以 case 字符串里的 v2/v3 为准。
    """
    meta = {"case": case, "group_label": case, "family_short": "", "family": ""}
    f = HIND_FIELDS_DIR / f"{VAR_TAG}_fields_{case}.nc"
    if not f.exists():
        return meta
    try:
        ds = xr.open_dataset(f)
        meta["group_label"] = ds.attrs.get("group_label", case)
        meta["family_short"] = ds.attrs.get("family_short", "")
        meta["family"] = ds.attrs.get("family", "")
        ds.close()
    except Exception:
        pass
    return meta


def nice_case_label(meta: dict, case: str, n_valid: int) -> str:
    """
    ✅ 标题里的 SP/LP/QP 现在按 case 字符串硬识别
    ✅ 名字里的 couple/nocouple 显示成 interactive/prescribed o3
    """
    gl_raw = meta.get("group_label", case)
    gl = pretty_exp_name(gl_raw)

    fam = infer_family_short_from_case(case)
    return f"{gl} ({fam}, n={n_valid})"


def load_ref_anom(year_int: int, season: str) -> xr.DataArray:
    if not REF_ANOM_FILE.exists():
        raise FileNotFoundError(f"Missing ref anomaly file: {REF_ANOM_FILE}")
    ds = xr.open_dataset(REF_ANOM_FILE)

    if REF_ANOM_VAR not in ds:
        ds.close()
        raise KeyError(f"Ref anomaly file missing variable '{REF_ANOM_VAR}'")

    ref = ds[REF_ANOM_VAR]  # (year,season,lat,lon)
    if year_int not in ref["year"].values:
        ds.close()
        raise KeyError(f"Reference missing year={year_int}. Available={ref['year'].values}")

    ref_y = ref.sel(year=year_int)
    ref_s = safe_sel_season(ref_y, season).load()
    ds.close()
    return ref_s


def load_hind_ens_anom(case: str, season: str) -> tuple[xr.DataArray, int]:
    f = HIND_ANOM_DIR / f"{VAR_TAG}_anom_fields_{case}.nc"
    if not f.exists():
        raise FileNotFoundError(f"Missing hindcast anomaly file: {f}")

    ds = xr.open_dataset(f)

    if HIND_ANOM_VAR not in ds:
        ds.close()
        raise KeyError(f"Hind anomaly file missing variable '{HIND_ANOM_VAR}'")

    z = ds[HIND_ANOM_VAR]  # (member,season,lat,lon)

    if "season_valid" in ds:
        sv = ds["season_valid"]
    else:
        sv = xr.DataArray(
            np.ones((z.sizes["member"], z.sizes["season"]), dtype=np.int8),
            dims=("member", "season"),
            coords={"member": z["member"], "season": z["season"]},
        )

    z_s  = safe_sel_season(z, season)
    sv_s = safe_sel_season(sv, season)

    z_s, sv_s = xr.align(z_s, sv_s, join="inner")
    valid = (sv_s == 1)
    n_valid = int(valid.sum().values)

    if n_valid == 0:
        nan2d = xr.full_like(z_s.isel(member=0, drop=True), np.nan)
        ds.close()
        return nan2d.load(), 0

    mem = sv_s["member"].where(valid, drop=True)
    ens = z_s.sel(member=mem).mean("member").load()
    ds.close()
    return ens, n_valid


def compute_common_vlim(fields_2d: list[xr.DataArray], q: float = 0.98) -> float:
    arrs = []
    for f in fields_2d:
        v = f.values
        v = v[np.isfinite(v)]
        if v.size:
            arrs.append(np.abs(v))
    if not arrs:
        return 1.0
    allabs = np.concatenate(arrs)
    vmax = float(np.quantile(allabs, q))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(allabs)) if np.isfinite(allabs).any() else 1.0
    return vmax


def set_circular_boundary(ax):
    theta = np.linspace(0, 2*np.pi, 400)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)


def plot_polar_map(ax, field2d: xr.DataArray, title: str, vlim: float):
    ax.set_title(title, fontsize=FS_TITLE, pad=3)
    ax.set_extent(POLAR_EXTENT, crs=DATA_CRS)
    ax.coastlines(linewidth=COASTLINE_LW)
    ax.add_feature(cfeature.BORDERS, linewidth=BORDER_LW, alpha=0.7)
    set_circular_boundary(ax)

    im = ax.pcolormesh(
        field2d["lon"], field2d["lat"], field2d,
        transform=DATA_CRS,
        cmap=CMAP,
        vmin=-vlim, vmax=vlim,
        shading="auto",
    )
    return im


def plot_one_figure(year_str: str, season: str, df_acc: pd.DataFrame) -> bool:
    season = normalize_season(season)
    year_int = int(year_str)

    # ---------- REF ----------
    try:
        ref2d = load_ref_anom(year_int, season)
    except Exception as e:
        print(f"[SKIP] {year_str} {season}: cannot load reference anomaly: {e}")
        return False

    # ---------- EXPS ----------
    cases_all = list_cases_for_year(year_str)
    exps = []
    for case in cases_all:
        try:
            ens2d, n_valid = load_hind_ens_anom(case, season)
        except Exception as e:
            print(f"  [WARN] {case} {season}: {e}")
            continue
        if n_valid <= 0:
            continue
        meta = load_case_meta(case)

        # ✅ 统一认为：除了 nocouple，其余都是 interactive o3（用于排序/标注都可）
        ozone_mode = infer_ozone_mode_from_case(case)

        exps.append({"case": case, "meta": meta, "ens": ens2d, "n_valid": n_valid, "ozone_mode": ozone_mode})

    if len(exps) < MIN_EXPS_PER_FIG:
        print(f"[SKIP] {year_str} {season}: only {len(exps)} valid experiments (<{MIN_EXPS_PER_FIG})")
        return False

    # ACC lookup
    acc_rows = df_acc[(df_acc["year_str"] == year_str) & (df_acc["season"] == season)].copy()
    acc_map = {r["case"]: r["ACC_ensmean"] for _, r in acc_rows.iterrows()}
    for e in exps:
        e["acc"] = float(acc_map.get(e["case"], np.nan))

    # ✅ sort：先按 prescribed/interactive，再按 family，再按 case
    def _sort_key(e):
        ozone_rank = 0 if e["ozone_mode"] == "interactive o3" else 1  # interactive 在前
        fam = infer_family_short_from_case(e["case"])
        return (ozone_rank, fam, e["case"])

    exps = sorted(exps, key=_sort_key)

    vlim = compute_common_vlim([ref2d] + [e["ens"] for e in exps], q=VLIM_Q)

    # ---------- LAYOUT ----------
    n_maps = 1 + len(exps)
    ncols = MAX_MAPS_PER_ROW
    nrows_maps = int(np.ceil(n_maps / ncols))

    # ✅ bar panel 更高
    bar_row_h = max(BAR_ROW_BASE, BAR_ROW_BASE + BAR_ROW_PER_BAR * max(0, len(exps) - 4))
    height_ratios = [1.0]*nrows_maps + [bar_row_h]

    fig = plt.figure(figsize=FIGSIZE)
    gs = fig.add_gridspec(
        nrows=nrows_maps + 1,
        ncols=ncols,
        height_ratios=height_ratios,
        hspace=0.18,
        wspace=0.06,
    )

    # ---------- MAPS ----------
    im_last = None
    idx = 0
    for k in range(n_maps):
        r = idx // ncols
        c = idx % ncols
        ax = fig.add_subplot(gs[r, c], projection=MAP_PROJ)

        if k == 0:
            ttl = f"REF (BWCN) anom — {year_str} {season}"
            im_last = plot_polar_map(ax, ref2d, ttl, vlim)
        else:
            e = exps[k-1]
            ttl = nice_case_label(e["meta"], e["case"], e["n_valid"])
            im_last = plot_polar_map(ax, e["ens"], ttl, vlim)

        idx += 1

    # remove unused map axes
    total_slots = nrows_maps * ncols
    for j in range(n_maps, total_slots):
        r = j // ncols
        c = j % ncols
        ax_empty = fig.add_subplot(gs[r, c])
        ax_empty.axis("off")

    # ---------- COLORBAR (RIGHT, VERTICAL) ----------
    cax = fig.add_axes([0.92, 0.28, 0.015, 0.55])
    cb = fig.colorbar(im_last, cax=cax, orientation="vertical")
    cb.ax.tick_params(labelsize=FS_TICK)
    cb.set_label(f"{VAR_TAG} anomaly (m)", fontsize=FS_LABEL)

    # ---------- ACC BAR (BOTTOM) ----------
    axb = fig.add_subplot(gs[-1, :])

    # ✅ bar 标签也用同样的替换逻辑
    labels = [pretty_exp_name(e["meta"].get("group_label", e["case"])) for e in exps]
    accs   = np.array([e["acc"] for e in exps], dtype=float)

    y = np.arange(len(labels), dtype=float) * BAR_Y_SPACING

    axb.barh(y, accs, height=BAR_HEIGHT)
    axb.set_yticks(y)
    axb.set_yticklabels(labels, fontsize=FS_TICK)
    axb.invert_yaxis()

    axb.axvline(0.0, linewidth=0.8, alpha=0.8)
    axb.set_xlim(-1.0, 1.0)
    axb.set_xlabel("ACC (ensemble mean)", fontsize=FS_LABEL)
    axb.set_title(f"ACC vs REF ({year_str} {season})", fontsize=FS_TITLE, pad=4)
    axb.tick_params(axis="x", labelsize=FS_TICK)

    # ✅ 再加一点上下留白，防止贴边挤
    if len(y) > 0:
        axb.set_ylim(y.max() + BAR_Y_SPACING*1.0, -BAR_Y_SPACING*1.0)

    # numeric labels at bar end
    for i, v in enumerate(accs):
        if np.isfinite(v):
            x = v + (0.03 if v >= 0 else -0.03)
            ha = "left" if v >= 0 else "right"
            axb.text(x, y[i], f"{v:.3f}", va="center", ha=ha, fontsize=FS_TICK)

    fig.suptitle(f"{VAR_TAG} anomaly patterns & ACC — {year_str} {season}", fontsize=11, y=0.985)

    out_png = OUT_FIG_DIR / f"lee_style_{VAR_TAG}_{year_str}_{season}_reversexy.png"
    out_pdf = OUT_FIG_DIR / f"lee_style_{VAR_TAG}_{year_str}_{season}_reversexy.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"[SAVE] {out_png}")
    return True


def main():
    print("=" * 100)
    print("[INFO] Lee-style plotting: polar projection unchanged + RIGHT vertical colorbar + spaced barh")
    print(f"  Target: {TARGET_PLEV_HPA} hPa ({PLEV_TARGET_PA} Pa)  -> {VAR_TAG}")
    print(f"  HIND_ANOM_DIR: {HIND_ANOM_DIR}")
    print(f"  REF_ANOM_FILE: {REF_ANOM_FILE}")
    print(f"  ACC_CSV      : {ACC_CSV}")
    print(f"  OUT_FIG_DIR  : {OUT_FIG_DIR}")
    print(f"  Expect vars  : hind='{HIND_ANOM_VAR}', ref='{REF_ANOM_VAR}'")
    print("=" * 100)

    df_acc = load_acc_table()

    print("\n[STEP] Example: 0008 MAM")
    plot_one_figure("0008", "MAM", df_acc)

    print("\n[STEP] Batch all")
    n_done, n_skip = 0, 0
    for y in TARGET_YEARS:
        for s in SEASONS:
            ok = plot_one_figure(y, s, df_acc)
            n_done += int(ok)
            n_skip += int(not ok)

    print("\n" + "=" * 100)
    print(f"[DONE] created: {n_done}, skipped: {n_skip}")
    print(f"[OUT ] {OUT_FIG_DIR}")
    print("=" * 100)


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations
import re
import numpy as np
import xarray as xr

# -------------------------------
# Helpers: case parsing
# -------------------------------
def parse_case(case_name: str) -> dict:
    """
    case_name examples:
      0008-02
      0008-02nocouple
      0008-02V2
      0008-02V3
    """
    m = re.match(r"^(?P<y>\d{4})-(?P<m>\d{2})(?P<suf>.*)$", case_name)
    if not m:
        raise ValueError(f"Bad case name: {case_name}")
    y = int(m.group("y"))
    mo = int(m.group("m"))
    suf = m.group("suf") or ""
    return {"case": case_name, "init_year": y, "init_month": mo, "suffix": suf}

def infer_family_short(case: str) -> str:
    c = case.lower()
    if "v2" in c:
        return "LP"
    if "v3" in c:
        return "QP"
    return "SP"

def infer_ozone_mode(case: str) -> str:
    c = case.lower()
    if ("nocouple" in c) or ("nocoupl" in c):
        return "prescribed o3"
    return "interactive o3"

# -------------------------------
# Model calendar arithmetic
# -------------------------------
def add_months(y: int, m: int, lead: int) -> tuple[int, int]:
    """
    lead=0 -> same (y,m)
    lead=1 -> next month
    """
    idx0 = (m - 1) + lead
    y2 = y + idx0 // 12
    m2 = (idx0 % 12) + 1
    return y2, m2

# -------------------------------
# Weights + pattern correlation
# -------------------------------
def coslat_weights(lat: xr.DataArray) -> np.ndarray:
    w = np.cos(np.deg2rad(lat.values))
    w = np.maximum(w, 0.0)
    return w.astype(np.float64)

def weighted_corr2d(a2d: np.ndarray, b2d: np.ndarray, w_lat: np.ndarray) -> float:
    """
    a2d,b2d: (lat,lon)
    w_lat: (lat,)
    """
    if a2d.ndim != 2 or b2d.ndim != 2:
        raise ValueError("a2d/b2d must be 2D (lat,lon).")
    if a2d.shape != b2d.shape:
        raise ValueError("a2d and b2d must have same shape.")
    lat_n, lon_n = a2d.shape
    w2d = np.repeat(w_lat.reshape(lat_n, 1), lon_n, axis=1)

    m = np.isfinite(a2d) & np.isfinite(b2d) & np.isfinite(w2d) & (w2d > 0)
    if m.sum() < 10:
        return np.nan

    aa = a2d[m]
    bb = b2d[m]
    ww = w2d[m]
    wsum = ww.sum()
    if wsum <= 0:
        return np.nan

    am = (ww * aa).sum() / wsum
    bm = (ww * bb).sum() / wsum

    a0 = aa - am
    b0 = bb - bm

    cov = (ww * a0 * b0).sum()
    va  = (ww * a0 * a0).sum()
    vb  = (ww * b0 * b0).sum()
    if va <= 0 or vb <= 0:
        return np.nan
    return float(cov / np.sqrt(va * vb))

# -------------------------------
# Vertical coordinate utilities
# -------------------------------
def unify_vertical_dim(da: xr.DataArray) -> xr.DataArray:
    if ("plev_2" in da.dims) and ("plev" not in da.dims):
        da = da.rename({"plev_2": "plev"})
    if ("lev" in da.dims) and ("plev" not in da.dims):
        da = da.rename({"lev": "plev"})
    if ("level" in da.dims) and ("plev" not in da.dims):
        da = da.rename({"level": "plev"})
    if ("plev" in da.dims) and ("plev_2" in da.dims):
        da = da.isel(plev_2=0, drop=True)
    return da

def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]  # (lev)
    hybm = ds["hybm"]  # (lev)
    P0   = ds["P0"]    # scalar Pa
    PS   = ds["PS"]    # (time, lat, lon)
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def interp_profile_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt: np.ndarray) -> xr.DataArray:
    """
    log(p) linear interpolation for each vertical column to target pressure levels.
    v_hyb, p_hyb: (..., lev), p in Pa
    returns: (..., plev)
    """
    p_tgt = np.asarray(p_tgt, float)
    nplev = int(p_tgt.size)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full((nplev,), np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)  # ascending pressure
        p_use = p_use[idx]
        v_use = v_use[idx]
        return np.interp(np.log(p_tgt), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={"plev": nplev},
    )
    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out
